# Shared Paper 2 split and model preprocessing

Freeze the existing 50/50 article split and create one common feature contract for all ranking models. Shared transformations produce author reception rates, discussion pace, prior reply composition, and leakage-safe centred reply depth. Recent-activity share and branch activity counts are excluded from the primary contract.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

from IPython.display import display

from commentgap_analysis.preprocessing import prepare_shared_model_data

SEED = int(os.getenv("COMMENTGAP_MODEL_SEED", "20260813"))
FEATURE_ROOT = Path(os.getenv("COMMENTGAP_FEATURE_ROOT", "model_output/selection_2025/features"))
MODEL_DATA_ROOT = Path(os.getenv("COMMENTGAP_MODEL_DATA_ROOT", "model_output/selection_2025/model_data"))
LEGACY_SPLIT = Path(os.getenv("COMMENTGAP_FROZEN_SPLIT", "model_output/selection_2025/xgboost_paper2/master_article_split.parquet"))
FORCE = os.getenv("COMMENTGAP_PREPROCESS_FORCE", "0").lower() in {"1", "true", "yes"}
assert LEGACY_SPLIT.exists(), f"Frozen Paper 2 split is missing: {LEGACY_SPLIT}"

{"feature_root": str(FEATURE_ROOT), "output_root": str(MODEL_DATA_ROOT), "frozen_split": str(LEGACY_SPLIT), "force": FORCE}

In [ ]:
result = prepare_shared_model_data(
    FEATURE_ROOT,
    MODEL_DATA_ROOT,
    frozen_split_path=LEGACY_SPLIT,
    seed=SEED,
    development_folds=5,
    force=FORCE,
)
result["rows"]

In [ ]:
display(result["article_split"]["split_role"].value_counts().rename("articles"))
display(result["split_balance"])
display(result["reply_depth_centers"])
for scope in ("root", "all"):
    features = result["feature_manifest"]["models"][scope]["features"]
    transformed = {
        "prior_reply_composition", "discussion_pace",
        "reply_depth_centered",
        "author_prior_30d_upvote_reception",
        "author_prior_30d_downvote_reception",
    }
    print(scope, len(features), [name for name in features if name in transformed])

## Downstream contract

All regression, XGBoost, and neural ranking workflows read the same v4 model-data contract. It contains discussion pace instead of total prior-comment counts, omits recent-activity share and branch activity, and substitutes fold-specific centred reply depth during development CV.